In [1]:
# Cell 0 | 导入依赖并设置路径

import hashlib
import json
import pickle
import sys

from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.model_selection import GroupKFold, KFold


def find_project_root(start_path="."):
    """向上查找同时包含 data 和 notebooks 的项目根目录。"""
    current_path = Path(start_path).resolve()

    for _ in range(20):
        if (
            (current_path / "data").exists()
            and
            (current_path / "notebooks").exists()
        ):
            return current_path

        if current_path.parent == current_path:
            break

        current_path = current_path.parent

    raise FileNotFoundError(
        "未找到项目根目录。请确认当前 notebook 位于：\n"
        r"D:\jupyter\IL_ML_Project\notebooks"
    )


def read_csv_robust(path):
    """依次尝试常见编码读取CSV。"""
    encodings = [
        "utf-8-sig",
        "utf-8",
        "gb18030",
    ]

    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(
                path,
                encoding=encoding,
            )
        except UnicodeDecodeError as error:
            last_error = error

    raise last_error


def calculate_sha256(path):
    """计算文件SHA256。"""
    sha256 = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            sha256.update(block)

    return sha256.hexdigest()


def find_json_value(obj, target_key):
    """递归查找JSON中的指定键。"""
    if isinstance(obj, dict):
        if target_key in obj:
            return obj[target_key]

        for value in obj.values():
            result = find_json_value(
                value,
                target_key,
            )

            if result is not None:
                return result

    elif isinstance(obj, list):
        for value in obj:
            result = find_json_value(
                value,
                target_key,
            )

            if result is not None:
                return result

    return None


PROJECT_ROOT = find_project_root(
    Path.cwd()
)

EXPERIMENT_ROOT = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
)

ML_READY_DIR = (
    EXPERIMENT_ROOT
    / "ml_ready"
)

REPRESENTATION_DIR = (
    EXPERIMENT_ROOT
    / "outputs"
    / "representation_compare"
)

MODEL_OUTPUT_ROOT = (
    EXPERIMENT_ROOT
    / "outputs"
    / "model_training_3models_cv"
)

SPLIT_DIR = (
    MODEL_OUTPUT_ROOT
    / "cv_splits"
)

SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# 输入文件
ML_READY_PATH = (
    ML_READY_DIR
    / "ml_ready_A.csv"
)

ML_READY_MANIFEST_PATH = (
    ML_READY_DIR
    / "ml_ready_A_manifest.json"
)

FEATURE_PATH = (
    REPRESENTATION_DIR
    / "mpnn_features_eta.npy"
)

FEATURE_ROW_MANIFEST_PATH = (
    REPRESENTATION_DIR
    / "mpnn_features_eta_row_manifest.csv"
)

FEATURE_ALIGNMENT_PATH = (
    REPRESENTATION_DIR
    / "mpnn_features_eta_alignment.npz"
)

FEATURE_METADATA_PATH = (
    EXPERIMENT_ROOT
    / "models"
    / "mpnn_feature_extractors"
    / "mpnn_feature_extractor_viscosity_metadata.json"
)


# 输出文件
CV_INDICES_PATH = (
    SPLIT_DIR
    / "global_cv_indices_eta_10point_holdout.pkl"
)

CV_ASSIGNMENTS_PATH = (
    SPLIT_DIR
    / "cv_split_assignments_eta_5fold.csv"
)

CV_AUDIT_PATH = (
    SPLIT_DIR
    / "cv_split_audit_eta_5fold.csv"
)

CV_INPUT_MANIFEST_PATH = (
    SPLIT_DIR
    / "cv_input_row_manifest_eta.csv"
)

CV_MANIFEST_PATH = (
    SPLIT_DIR
    / "cv_split_manifest_eta_5fold.json"
)


required_files = [
    ML_READY_PATH,
    ML_READY_MANIFEST_PATH,
    FEATURE_PATH,
    FEATURE_ROW_MANIFEST_PATH,
    FEATURE_ALIGNMENT_PATH,
    FEATURE_METADATA_PATH,
]

missing_files = [
    path
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "缺少以下输入文件：\n"
        + "\n".join(
            str(path)
            for path in missing_files
        )
    )


experiment_text = str(
    EXPERIMENT_ROOT.resolve()
).lower()

if (
    "small_paper_data" not in experiment_text
    or
    "viscosity_10point_holdout"
    not in experiment_text
):
    raise RuntimeError(
        "实验目录异常，已停止运行，"
        "防止覆盖大论文数据。"
    )


RANDOM_SEED = 42
N_SPLITS = 5

SPLIT_ORDER = [
    "Data_split",
    "IL_split",
    "Cation_split",
    "Anion_split",
]


print("python              =", sys.executable)
print("PROJECT_ROOT        =", PROJECT_ROOT)
print("EXPERIMENT_ROOT     =", EXPERIMENT_ROOT)
print("SPLIT_DIR           =", SPLIT_DIR)

print("\n输入文件：")
print("ml_ready_A          =", ML_READY_PATH)
print("MPNN features       =", FEATURE_PATH)
print("feature row manifest=", FEATURE_ROW_MANIFEST_PATH)

print("\n外部10点：")
print("本 notebook 不读取外部10点文件。")

python              = D:\conda\envs\rdkit-env\python.exe
PROJECT_ROOT        = D:\jupyter\IL_ML_Project
EXPERIMENT_ROOT     = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout
SPLIT_DIR           = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv\cv_splits

输入文件：
ml_ready_A          = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\ml_ready\ml_ready_A.csv
MPNN features       = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\representation_compare\mpnn_features_eta.npy
feature row manifest= D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\representation_compare\mpnn_features_eta_row_manifest.csv

外部10点：
本 notebook 不读取外部10点文件。


In [2]:
# Cell 1 | 读取并核对建模数据与MPNN特征

df_model = read_csv_robust(
    ML_READY_PATH
)

df_feature_rows = read_csv_robust(
    FEATURE_ROW_MANIFEST_PATH
)

X = np.load(
    FEATURE_PATH,
)

feature_alignment = np.load(
    FEATURE_ALIGNMENT_PATH,
)

with open(
    ML_READY_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as file:
    ml_ready_manifest = json.load(file)

with open(
    FEATURE_METADATA_PATH,
    "r",
    encoding="utf-8",
) as file:
    feature_metadata = json.load(file)


df_model.columns = (
    df_model.columns
    .astype(str)
    .str.strip()
)

df_feature_rows.columns = (
    df_feature_rows.columns
    .astype(str)
    .str.strip()
)


required_columns = [
    "IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df_model.columns
]

if missing_columns:
    raise KeyError(
        "ml_ready_A.csv缺少必要字段："
        f"{missing_columns}"
    )


for column in [
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]:
    df_model[column] = pd.to_numeric(
        df_model[column],
        errors="coerce",
    )


if X.ndim != 2:
    raise ValueError(
        f"MPNN特征不是二维矩阵：{X.shape}"
    )

if X.shape[1] != 130:
    raise ValueError(
        "MPNN特征维度不是130："
        f"{X.shape[1]}"
    )

if len(df_model) != X.shape[0]:
    raise ValueError(
        "ml_ready_A.csv与MPNN特征行数不一致：\n"
        f"ml_ready rows = {len(df_model)}\n"
        f"feature rows  = {X.shape[0]}"
    )

if len(df_feature_rows) != X.shape[0]:
    raise ValueError(
        "特征行清单与MPNN特征行数不一致：\n"
        f"row manifest rows = {len(df_feature_rows)}\n"
        f"feature rows      = {X.shape[0]}"
    )

if not np.isfinite(X).all():
    raise ValueError(
        "MPNN特征矩阵存在NaN或无穷值。"
    )

if df_model[required_columns].isna().any().any():
    missing_summary = (
        df_model[required_columns]
        .isna()
        .sum()
    )

    raise ValueError(
        "建模数据必要字段存在缺失：\n"
        f"{missing_summary}"
    )

if not np.isfinite(
    df_model[
        [
            "T_rep",
            "P_rep",
            "eta",
            "lg_eta",
        ]
    ].to_numpy(dtype=float)
).all():
    raise ValueError(
        "温度、压力、黏度或lg_eta中存在无穷值。"
    )

if (df_model["eta"] <= 0).any():
    raise ValueError(
        "eta中存在非正值。"
    )


lg_eta_recalculated = np.log10(
    df_model["eta"].to_numpy(dtype=float)
)

max_lg_difference = float(
    np.max(
        np.abs(
            df_model["lg_eta"].to_numpy(dtype=float)
            - lg_eta_recalculated
        )
    )
)

if max_lg_difference > 1e-10:
    raise ValueError(
        "lg_eta与log10(eta)不一致：\n"
        f"最大差值={max_lg_difference:.12e}"
    )


# 核对特征行清单与ml_ready数据的顺序
compare_string_columns = [
    "IL_key",
    "cation_smiles",
    "anion_smiles",
]

for column in compare_string_columns:
    if column in df_feature_rows.columns:
        left_values = (
            df_model[column]
            .astype(str)
            .to_numpy()
        )

        right_values = (
            df_feature_rows[column]
            .astype(str)
            .to_numpy()
        )

        if not np.array_equal(
            left_values,
            right_values,
        ):
            raise ValueError(
                f"特征行清单中的{column}"
                "与ml_ready_A.csv顺序不一致。"
            )


compare_numeric_columns = [
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]

for column in compare_numeric_columns:
    if column in df_feature_rows.columns:
        left_values = (
            df_model[column]
            .to_numpy(dtype=float)
        )

        right_values = pd.to_numeric(
            df_feature_rows[column],
            errors="coerce",
        ).to_numpy(dtype=float)

        if not np.allclose(
            left_values,
            right_values,
            rtol=0.0,
            atol=1e-12,
            equal_nan=True,
        ):
            raise ValueError(
                f"特征行清单中的{column}"
                "与ml_ready_A.csv顺序不一致。"
            )


if "source_row_idx" in df_feature_rows.columns:
    source_row_idx = pd.to_numeric(
        df_feature_rows["source_row_idx"],
        errors="raise",
    ).astype(int).to_numpy()

else:
    source_row_idx = np.arange(
        len(df_model),
        dtype=int,
    )


if len(np.unique(source_row_idx)) != len(source_row_idx):
    raise ValueError(
        "source_row_idx存在重复。"
    )


external_holdout_used = find_json_value(
    feature_metadata,
    "external_holdout_used",
)

if external_holdout_used is True:
    raise RuntimeError(
        "MPNN metadata显示外部留出点参与了特征预训练，"
        "已停止运行。"
    )


df_cv = df_model.copy()
df_cv.insert(
    0,
    "source_row_idx",
    source_row_idx,
)

df_cv.to_csv(
    CV_INPUT_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)


print("输入数据与MPNN特征严格对齐 ✓")
print("model rows          =", len(df_cv))
print("feature shape       =", X.shape)
print("unique ILs          =", df_cv["IL_key"].nunique())
print("unique cations      =", df_cv["cation_smiles"].nunique())
print("unique anions       =", df_cv["anion_smiles"].nunique())
print("max lg difference   =", max_lg_difference)
print("external holdout used =", external_holdout_used)
print("alignment keys      =", feature_alignment.files)
print("saved:", CV_INPUT_MANIFEST_PATH)

输入数据与MPNN特征严格对齐 ✓
model rows          = 16216
feature shape       = (16216, 130)
unique ILs          = 1295
unique cations      = 583
unique anions       = 202
max lg difference   = 1.7763568394002505e-15
external holdout used = None
alignment keys      = ['feature_row_idx', 'source_row_idx', 'graph_list_idx', 'internal_train_feature_row_idx', 'internal_val_feature_row_idx']
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv\cv_splits\cv_input_row_manifest_eta.csv


In [3]:
# Cell 2 | 生成四种5折交叉验证索引

n_samples = len(df_cv)

all_indices = np.arange(
    n_samples,
    dtype=int,
)

group_arrays = {
    "IL_split": (
        df_cv["IL_key"]
        .astype(str)
        .to_numpy()
    ),
    "Cation_split": (
        df_cv["cation_smiles"]
        .astype(str)
        .to_numpy()
    ),
    "Anion_split": (
        df_cv["anion_smiles"]
        .astype(str)
        .to_numpy()
    ),
}


cv_indices = {
    "eta": {},
}

df_assignments = df_cv[
    [
        "source_row_idx",
        "IL_key",
        "cation_smiles",
        "anion_smiles",
        "T_rep",
        "P_rep",
        "eta",
        "lg_eta",
    ]
].copy()

audit_rows = []


for split_name in SPLIT_ORDER:
    cv_indices["eta"][split_name] = {}

    assignment_column = (
        f"{split_name}_test_fold"
    )

    df_assignments[
        assignment_column
    ] = -1

    if split_name == "Data_split":
        splitter = KFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=RANDOM_SEED,
        )

        split_iterator = splitter.split(
            all_indices
        )

        group_values = None

    else:
        group_values = group_arrays[
            split_name
        ]

        n_unique_groups = len(
            np.unique(group_values)
        )

        if n_unique_groups < N_SPLITS:
            raise ValueError(
                f"{split_name}的唯一分组数"
                f"不足{N_SPLITS}："
                f"{n_unique_groups}"
            )

        splitter = GroupKFold(
            n_splits=N_SPLITS,
        )

        split_iterator = splitter.split(
            all_indices,
            groups=group_values,
        )

    for fold, (
        train_idx,
        test_idx,
    ) in enumerate(split_iterator):
        train_idx = np.asarray(
            train_idx,
            dtype=int,
        )

        test_idx = np.asarray(
            test_idx,
            dtype=int,
        )

        overlap = np.intersect1d(
            train_idx,
            test_idx,
        )

        if len(overlap) != 0:
            raise RuntimeError(
                f"{split_name} Fold {fold}"
                "的训练集和测试集存在重叠。"
            )

        if (
            len(train_idx)
            + len(test_idx)
            != n_samples
        ):
            raise RuntimeError(
                f"{split_name} Fold {fold}"
                "未覆盖全部样本。"
            )

        group_overlap_count = 0
        train_group_count = np.nan
        test_group_count = np.nan

        if group_values is not None:
            train_groups = np.unique(
                group_values[train_idx]
            )

            test_groups = np.unique(
                group_values[test_idx]
            )

            group_overlap_count = len(
                np.intersect1d(
                    train_groups,
                    test_groups,
                )
            )

            train_group_count = len(
                train_groups
            )

            test_group_count = len(
                test_groups
            )

            if group_overlap_count != 0:
                raise RuntimeError(
                    f"{split_name} Fold {fold}"
                    "存在分组泄露。"
                )

        cv_indices["eta"][
            split_name
        ][fold] = {
            "train": train_idx.tolist(),
            "test": test_idx.tolist(),
        }

        df_assignments.loc[
            test_idx,
            assignment_column,
        ] = fold

        y_train = (
            df_cv.loc[
                train_idx,
                "lg_eta",
            ]
            .to_numpy(dtype=float)
        )

        y_test = (
            df_cv.loc[
                test_idx,
                "lg_eta",
            ]
            .to_numpy(dtype=float)
        )

        audit_rows.append(
            {
                "Property": "eta",
                "Target": "lg_eta",
                "Split": split_name,
                "Fold": fold,
                "Train_rows": len(
                    train_idx
                ),
                "Test_rows": len(
                    test_idx
                ),
                "Total_rows": (
                    len(train_idx)
                    + len(test_idx)
                ),
                "Index_overlap_count": len(
                    overlap
                ),
                "Train_group_count": (
                    train_group_count
                ),
                "Test_group_count": (
                    test_group_count
                ),
                "Group_overlap_count": (
                    group_overlap_count
                ),
                "Train_lg_eta_mean": float(
                    np.mean(y_train)
                ),
                "Train_lg_eta_std": float(
                    np.std(
                        y_train,
                        ddof=1,
                    )
                ),
                "Test_lg_eta_mean": float(
                    np.mean(y_test)
                ),
                "Test_lg_eta_std": float(
                    np.std(
                        y_test,
                        ddof=1,
                    )
                ),
                "Test_lg_eta_min": float(
                    np.min(y_test)
                ),
                "Test_lg_eta_max": float(
                    np.max(y_test)
                ),
            }
        )


for split_name in SPLIT_ORDER:
    assignment_column = (
        f"{split_name}_test_fold"
    )

    fold_counts = (
        df_assignments[
            assignment_column
        ]
        .value_counts()
        .sort_index()
    )

    if set(fold_counts.index) != set(
        range(N_SPLITS)
    ):
        raise RuntimeError(
            f"{split_name}的测试折编号异常。"
        )

    if (
        df_assignments[
            assignment_column
        ] < 0
    ).any():
        raise RuntimeError(
            f"{split_name}存在未分配测试折的样本。"
        )


df_audit = pd.DataFrame(
    audit_rows
)

with open(
    CV_INDICES_PATH,
    "wb",
) as file:
    pickle.dump(
        cv_indices,
        file,
    )

df_assignments.to_csv(
    CV_ASSIGNMENTS_PATH,
    index=False,
    encoding="utf-8-sig",
)

df_audit.to_csv(
    CV_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)


print("四种5折交叉验证索引生成完成 ✓")
print("saved:", CV_INDICES_PATH)
print("saved:", CV_ASSIGNMENTS_PATH)
print("saved:", CV_AUDIT_PATH)

display(
    df_audit
)

四种5折交叉验证索引生成完成 ✓
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv\cv_splits\global_cv_indices_eta_10point_holdout.pkl
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv\cv_splits\cv_split_assignments_eta_5fold.csv
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv\cv_splits\cv_split_audit_eta_5fold.csv


,Property,Target,Split,Fold,Train_rows,Test_rows,Total_rows,Index_overlap_count,Train_group_count,Test_group_count,Group_overlap_count,Train_lg_eta_mean,Train_lg_eta_std,Test_lg_eta_mean,Test_lg_eta_std,Test_lg_eta_min,Test_lg_eta_max
0,eta,lg_eta,Data_split,0,12972,3244,16216,0,NaN,NaN,0,1.899874,0.826363,1.906949,0.873428,-0.107905,12.113943
1,eta,lg_eta,Data_split,1,12973,3243,16216,0,NaN,NaN,0,1.903023,0.828662,1.894352,0.864671,-0.657577,12.301030
2,eta,lg_eta,Data_split,2,12973,3243,16216,0,NaN,NaN,0,1.897607,0.829315,1.916021,0.862042,-1.000000,11.707570
3,eta,lg_eta,Data_split,3,12973,3243,16216,0,NaN,NaN,0,1.909272,0.855953,1.869356,0.749991,-0.744727,9.301030
4,eta,lg_eta,Data_split,4,12973,3243,16216,0,NaN,NaN,0,1.896670,0.839156,1.919766,0.822957,-0.920819,10.380211
5,eta,lg_eta,IL_split,0,12972,3244,16216,0,1036.0,259.0,0,1.905040,0.829511,1.886293,0.861272,-0.275724,10.908485
6,eta,lg_eta,IL_split,1,12973,3243,16216,0,1036.0,259.0,0,1.905420,0.857517,1.884763,0.743437,0.056905,5.146128
7,eta,lg_eta,IL_split,2,12973,3243,16216,0,1036.0,259.0,0,1.872521,0.767443,2.016372,1.059305,0.262451,12.301030
8,eta,lg_eta,IL_split,3,12973,3243,16216,0,1036.0,259.0,0,1.914288,0.859791,1.849289,0.730775,-1.000000,5.235528
9,eta,lg_eta,IL_split,4,12973,3243,16216,0,1036.0,259.0,0,1.909178,0.861128,1.869734,0.725923,-0.045757,6.017033


In [4]:
# Cell 3 | 最终核查并保存索引manifest

with open(
    CV_INDICES_PATH,
    "rb",
) as file:
    check_indices = pickle.load(file)


if set(
    check_indices["eta"].keys()
) != set(SPLIT_ORDER):
    raise RuntimeError(
        "保存后的划分方式不完整。"
    )


for split_name in SPLIT_ORDER:
    folds = check_indices[
        "eta"
    ][split_name]

    if len(folds) != N_SPLITS:
        raise RuntimeError(
            f"{split_name}的折数不是5。"
        )

    test_coverage = []

    for fold in range(N_SPLITS):
        train_idx = np.asarray(
            folds[fold]["train"],
            dtype=int,
        )

        test_idx = np.asarray(
            folds[fold]["test"],
            dtype=int,
        )

        if len(
            np.intersect1d(
                train_idx,
                test_idx,
            )
        ) != 0:
            raise RuntimeError(
                f"{split_name} Fold {fold}"
                "存在索引重叠。"
            )

        test_coverage.extend(
            test_idx.tolist()
        )

    test_coverage = np.asarray(
        test_coverage,
        dtype=int,
    )

    unique_test_indices, counts = (
        np.unique(
            test_coverage,
            return_counts=True,
        )
    )

    if len(unique_test_indices) != len(
        df_cv
    ):
        raise RuntimeError(
            f"{split_name}的5个测试折"
            "没有覆盖全部样本。"
        )

    if not np.all(counts == 1):
        raise RuntimeError(
            f"{split_name}中存在样本"
            "被重复分入测试集。"
        )


manifest = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),

    "task": (
        "viscosity_only_"
        "four_split_5fold_cv_indices"
    ),

    "project_root": str(
        PROJECT_ROOT
    ),

    "experiment_root": str(
        EXPERIMENT_ROOT
    ),

    "property": "viscosity",

    "target": "lg_eta",

    "representation": "MPNN",

    "fingerprints_used": False,

    "external_holdout_file_read": False,

    "external_holdout_used_for_split": False,

    "random_seed": RANDOM_SEED,

    "n_splits": N_SPLITS,

    "split_order": SPLIT_ORDER,

    "split_definitions": {
        "Data_split": (
            "KFold with shuffled data points"
        ),
        "IL_split": (
            "GroupKFold grouped by IL_key"
        ),
        "Cation_split": (
            "GroupKFold grouped by cation_smiles"
        ),
        "Anion_split": (
            "GroupKFold grouped by anion_smiles"
        ),
    },

    "counts": {
        "model_rows": int(
            len(df_cv)
        ),
        "feature_rows": int(
            X.shape[0]
        ),
        "feature_dim": int(
            X.shape[1]
        ),
        "unique_ILs": int(
            df_cv["IL_key"].nunique()
        ),
        "unique_cations": int(
            df_cv[
                "cation_smiles"
            ].nunique()
        ),
        "unique_anions": int(
            df_cv[
                "anion_smiles"
            ].nunique()
        ),
    },

    "input_files": {
        "ml_ready_A": str(
            ML_READY_PATH
        ),
        "mpnn_features": str(
            FEATURE_PATH
        ),
        "feature_row_manifest": str(
            FEATURE_ROW_MANIFEST_PATH
        ),
        "feature_metadata": str(
            FEATURE_METADATA_PATH
        ),
    },

    "input_hashes": {
        "ml_ready_A_sha256": (
            calculate_sha256(
                ML_READY_PATH
            )
        ),
        "mpnn_features_sha256": (
            calculate_sha256(
                FEATURE_PATH
            )
        ),
        "feature_row_manifest_sha256": (
            calculate_sha256(
                FEATURE_ROW_MANIFEST_PATH
            )
        ),
    },

    "outputs": {
        "cv_indices": str(
            CV_INDICES_PATH
        ),
        "cv_assignments": str(
            CV_ASSIGNMENTS_PATH
        ),
        "cv_audit": str(
            CV_AUDIT_PATH
        ),
        "cv_input_row_manifest": str(
            CV_INPUT_MANIFEST_PATH
        ),
    },
}


with open(
    CV_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        ensure_ascii=False,
        indent=2,
    )


print("=" * 82)
print("全部交叉验证索引检查通过 ✓")
print("=" * 82)

print("model rows          =", len(df_cv))
print("feature shape       =", X.shape)
print("target              = lg_eta")
print("representation      = MPNN")
print("fingerprints used   = False")
print("external holdout used =", False)

print("\n四种划分：")
for split_name in SPLIT_ORDER:
    print(
        split_name,
        "folds =",
        len(
            check_indices[
                "eta"
            ][split_name]
        ),
    )

print("\n输出文件完整路径：")
print(CV_INDICES_PATH)
print(CV_ASSIGNMENTS_PATH)
print(CV_AUDIT_PATH)
print(CV_INPUT_MANIFEST_PATH)
print(CV_MANIFEST_PATH)

全部交叉验证索引检查通过 ✓
model rows          = 16216
feature shape       = (16216, 130)
target              = lg_eta
representation      = MPNN
fingerprints used   = False
external holdout used = False

四种划分：
Data_split folds = 5
IL_split folds = 5
Cation_split folds = 5
Anion_split folds = 5

输出文件完整路径：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv\cv_splits\global_cv_indices_eta_10point_holdout.pkl
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv\cv_splits\cv_split_assignments_eta_5fold.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv\cv_splits\cv_split_audit_eta_5fold.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv\cv_splits\cv_input_row_manifest_eta.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv\cv_splits\cv_split_